# pmcache &mdash; Perceptual MM-Cache for Video LLMs
### Hackathon track: **Acceleration**

**The problem.** vLLM's `MultiModalHasher` hashes raw pixels, so two
nearly-identical adjacent frames produce two different `mm_hash`es. The
vision encoder runs once per frame, LMCache stores one KV block per
frame, and TTFT scales linearly with frame count.

**The fix.** `pmcache` (perceptual mm-cache) tags perceptually-similar
frames with the **same** EXIF `ImageID` UUID *before* they reach vLLM:

1. **Tier&nbsp;1 &mdash; pHash + BK-tree** &mdash; find anchor candidates within
   Hamming&nbsp;&le;&nbsp;`K`. CPU-only, microseconds.
2. **Tier&nbsp;2 &mdash; DINOv2-small cosine** &mdash; verify visual identity at
   cosine&nbsp;&ge;&nbsp;`&tau;`. ~30&nbsp;ms/frame on a T4.

vLLM's hasher uses our UUID &rarr; LMCache hits its existing exact-match
path &rarr; **the vision encoder runs once per anchor, not per frame**.

**What this notebook will show:**
- &#9889; **TTFT acceleration** &mdash; measured wall-clock improvement on MVBench.
- &#128190; **KV memory reduction** &mdash; fewer unique vision-token chunks.
- &#127919; **Zero accuracy loss** &mdash; DINOv2 threshold guarantees fidelity.

### Architecture

```
                       sample_frames(video, fps)
                                   |
                                   v
   +-------------+    +-----------------------------+
   |  PIL Image  |--->|  PerceptualMMCache          |
   +-------------+    |                             |
                      |  Tier 1: pHash + BK-tree    |  candidates w/in K Hamming
                      |  Tier 2: DINOv2 cosine >= t |  verify visual identity
                      |                             |
                      |  -> EXIF ImageID = UUID     |  hit:  anchor UUID
                      |                             |  miss: fresh UUID + new bucket
                      +-------------+---------------+
                                    |
                                    v
                  +----------------------------------+
                  | vLLM MultiModalHasher reads EXIF |
                  | -> mm_hash = UUID bytes          |
                  +----------------------------------+
                                    |
                                    v
                          LMCache exact-match path
                          (KV blocks reused for all
                           aliased frames)
```

## 0 &middot; Configuration

In [ ]:
# ====================== EDIT THESE ======================
REPO_URL  = "https://github.com/dheerajmr01/perceptual-mmcache.git"
BRANCH    = "main"
MODEL     = "Qwen/Qwen3-VL-2B-Instruct"          # 256K ctx, A100-40G verified
WORKSPACE = "/content/drive/MyDrive/pmcache"

# Perceptual-cache hyperparameters
TAU = 0.98   # DINOv2 cosine threshold (higher = more conservative)
K   = 5      # pHash Hamming cap for BK-tree candidates
FPS = 1.0    # frame sample rate (higher FPS = more redundancy = bigger win)

# Benchmark dataset (MVBench tasks with HF download support)
N_VIDEOS      = 10
MVBENCH_TASKS = ["object_existence", "object_interaction",
                 "action_sequence", "scene_transition"]
# =========================================================

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PMCACHE_WORKSPACE"] = WORKSPACE
os.environ["PMCACHE_DEVICE"]    = "cuda"

## 1 &middot; GPU sanity check

In [ ]:
!nvidia-smi

## 2 &middot; Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3 &middot; Install dependencies

In [ ]:
%%capture
!pip install -q --upgrade pip

# vllm + lmcache + HF stack — pip resolves a compatible torch.
# flashinfer-cubin ships pre-compiled CUDA kernels (avoids JIT crash on
# A100 sm_80 with newer flashinfer wheels).
!pip install -q -U \
    vllm \
    lmcache \
    transformers \
    accelerate \
    hf_transfer \
    flashinfer-python \
    flashinfer-cubin

# Pure-Python + plotting + dataset loaders.
!pip install -q \
    imagehash pybktree opencv-python-headless pillow \
    matplotlib pandas tqdm seaborn pytest \
    datasets huggingface_hub yt-dlp

### &#9888;&#65039; Restart runtime before continuing
vLLM only picks up its native dependencies after a fresh interpreter.
Run the next cell, then **Runtime &rarr; Restart runtime**, then
continue from section 4 onward.

In [ ]:
# ===== RESTART RUNTIME — then continue from section 4 =====
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

---

## 4 &middot; Post-restart &mdash; re-apply config + verify stack

In [ ]:
# Keep these in sync with section 0.
REPO_URL  = "https://github.com/dheerajmr01/perceptual-mmcache.git"
BRANCH    = "main"
MODEL     = "Qwen/Qwen3-VL-2B-Instruct"
WORKSPACE = "/content/drive/MyDrive/pmcache"
TAU, K, FPS = 0.98, 5, 1.0
N_VIDEOS      = 10
MVBENCH_TASKS = ["object_existence", "object_interaction",
                 "action_sequence", "scene_transition"]

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PMCACHE_WORKSPACE"] = WORKSPACE
os.environ["PMCACHE_DEVICE"]    = "cuda"

from google.colab import drive
drive.mount('/content/drive')

import vllm, lmcache, torch, transformers
print(f"vllm:         {vllm.__version__}")
print(f"lmcache:      {getattr(lmcache, '__version__', '(installed)')}")
print(f"torch:        {torch.__version__}  (CUDA: {torch.cuda.is_available()})")
print(f"transformers: {transformers.__version__}")

## 5 &middot; Clone repo + install pmcache

In [ ]:
%cd /content
!rm -rf perceptual-mmcache
!git clone -b {BRANCH} {REPO_URL}
%cd /content/perceptual-mmcache
!pip install -q -e ".[mvbench,eval]"

## 6 &middot; Initialize paths + run CPU sanity tests

In [ ]:
from eval import paths
paths.set_workspace(WORKSPACE)
paths.ensure_dirs()
print(f"VIDEOS_DIR  = {paths.VIDEOS_DIR}")
print(f"QA_FILE     = {paths.QA_FILE}")
print(f"RESULTS_DIR = {paths.RESULTS_DIR}")
print(f"REPORT_DIR  = {paths.REPORT_DIR}")

# Sanity-check the harness before the long GPU runs.
!pytest tests/ -q --tb=short

## 7 &middot; Prepare MVBench subset

We pull metadata from `OpenGVLab/MVBench` for a curated set of
cache-friendly tasks (Charades + STAR + ScanNet sources have the most
visual redundancy across adjacent frames). Videos are downloaded from
the HF dataset repo &mdash; no YouTube/yt-dlp involved.

If MVBench download fails (gated config, network), we fall back to the
Video-MME loader. Either way, the rest of the notebook is identical.

In [ ]:
import os

video_files = sorted(f for f in os.listdir(paths.VIDEOS_DIR)
                     if f.lower().endswith(('.mp4', '.mov', '.webm', '.mkv')))

if len(video_files) >= 5 and os.path.exists(paths.QA_FILE):
    print(f"[skip] already have {len(video_files)} videos + qa.jsonl")
else:
    try:
        from eval.datasets.prepare_mvbench import main as prepare_mvbench
        prepare_mvbench(
            output_dir=paths.VIDEOS_DIR,
            n=N_VIDEOS,
            tasks=MVBENCH_TASKS,
            max_questions_per_video=5,
        )
    except Exception as e:
        print(f"\n[!] MVBench prep failed: {e}\n[->] falling back to Video-MME")
        from eval.datasets.prepare_videomme import main as prepare_videomme
        prepare_videomme(output_dir=paths.VIDEOS_DIR, n=N_VIDEOS,
                         duration="short", max_height=360)

video_files = sorted(f for f in os.listdir(paths.VIDEOS_DIR)
                     if f.lower().endswith(('.mp4', '.mov', '.webm', '.mkv')))
print(f"\nVideos ready ({len(video_files)}):")
for v in video_files:
    sz = os.path.getsize(os.path.join(paths.VIDEOS_DIR, v)) / 1e6
    print(f"  - {v}  ({sz:.1f} MB)")

## 8 &middot; Visualize the redundancy pmcache exploits

Before running the benchmark: let's *see* the structure pmcache is
betting on. We sample one video at `FPS=1`, then compute the
frame&times;frame pHash Hamming-distance matrix. **Dark cells = similar
frames; bright = different.** A strong dark band along the diagonal
is exactly the run-of-near-duplicates pattern pmcache collapses.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from eval.utils import sample_frames
from pmcache.hashing import phash

sample_video = paths.VIDEOS_DIR / video_files[0]
frames = sample_frames(sample_video, fps=FPS, max_frames=40)
hashes = [phash(f) for f in frames]

def ham(a: int, b: int) -> int:
    return bin(a ^ b).count('1')

mat = np.array([[ham(a, b) for b in hashes] for a in hashes])

fig, axes = plt.subplots(1, 2, figsize=(14, 5),
                         gridspec_kw={'width_ratios': [3, 2]})

sns.heatmap(mat, ax=axes[0], cmap='magma',
            cbar_kws={'label': 'pHash Hamming distance'})
axes[0].set_title(f'Pairwise pHash distance\n{sample_video.name}  ({len(frames)} frames @ {FPS} fps)')
axes[0].set_xlabel('frame index'); axes[0].set_ylabel('frame index')

iu = np.triu_indices(len(hashes), k=1)
dists = mat[iu]
axes[1].hist(dists, bins=range(0, 65, 2), color='#4a90e2', edgecolor='white')
axes[1].axvline(K, color='red', ls='--', lw=2, label=f'K = {K} (BK-tree cap)')
axes[1].set_xlabel('pairwise pHash distance')
axes[1].set_ylabel('# of frame pairs')
axes[1].set_title('Distance distribution')
axes[1].legend()

plt.tight_layout(); plt.show()

within_k = int((dists <= K).sum())
print(f"\nPairs within K={K}: {within_k} / {len(dists)} "
      f"({within_k/len(dists)*100:.1f}%)")
print(f"-> ~{within_k} aliasable candidate pairs in this single video alone")

## 9 &middot; Visualize the aliasing pmcache produces

Now we run the *full* two-tier shim (pHash + DINOv2) on those frames.
Each tile below is a sampled frame, with its border colored by the
**anchor UUID** the shim assigned. **Same color = same `mm_hash` =
LMCache hit.** The compression ratio printed beneath is the per-video
reduction in unique vision-token chunks.

In [ ]:
from pmcache.config import PMCacheConfig
from pmcache.lmcache_shim import PerceptualMMCache, _existing_uuid

shim = PerceptualMMCache(PMCacheConfig(enabled=True, tau=TAU, k=K))

tagged = [shim.prepare_image(f) for f in frames]
uuids = [(_existing_uuid(t).hex if _existing_uuid(t) else None) for t in tagged]
unique_anchors = sorted({u for u in uuids if u})

palette = plt.cm.tab20(np.linspace(0, 1, max(len(unique_anchors), 2)))
color_for = {u: palette[i] for i, u in enumerate(unique_anchors)}

n = len(frames)
cols = min(n, 10)
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.6, rows * 1.6))
axes = np.atleast_2d(axes)
for i, (frame, anchor) in enumerate(zip(frames, uuids)):
    ax = axes[i // cols, i % cols]
    ax.imshow(frame.resize((128, 128)))
    color = color_for.get(anchor, (0.5, 0.5, 0.5, 1.0))
    for spine in ax.spines.values():
        spine.set_edgecolor(color); spine.set_linewidth(5)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel(f"#{i}", fontsize=8)
for j in range(n, rows * cols):
    axes[j // cols, j % cols].axis('off')

fig.suptitle(
    f"{sample_video.name}: {n} frames -> {len(unique_anchors)} anchors "
    f"(same color = aliased to same UUID = LMCache hit)",
    fontsize=11,
)
plt.tight_layout(); plt.show()

m = shim.metrics
print(f"\nframes seen:       {m.frames_seen}")
print(f"Tier-1 hits:       {m.tier1_hits}  (pHash matched a candidate)")
print(f"Tier-2 hits:       {m.tier2_hits}  (cosine >= {TAU} -> aliased)")
print(f"Tier-2 rejects:    {m.tier2_rejects}  (false-positive avoided)")
print(f"new anchors:       {m.misses}")
print(f"\ncompression: {n} frames -> {len(unique_anchors)} anchors "
      f"({(1 - len(unique_anchors)/n)*100:.1f}% reduction)")

## 10 &middot; VLM smoke test

Load `MODEL` once and run a single inference. Catches OOM /
version-mismatch failures before committing to the full benchmark.

In [ ]:
from eval.utils import smoke_test_vlm
smoke_test_vlm(MODEL, WORKSPACE)

## 11 &middot; Run 1 &mdash; **Baseline** (bytewise mm_hash)

vLLM hashes raw pixel bytes. Every frame is unique; the vision encoder
runs once per frame; LMCache stores one KV block per frame. This is
the &quot;before&quot; picture.

In [ ]:
from eval.run_baseline import run_baseline_benchmark

baseline_summary = run_baseline_benchmark(
    videos_dir=paths.VIDEOS_DIR,
    qa_file=paths.QA_FILE,
    output_path=paths.BASELINE_PATH,
    model=MODEL,
    fps=FPS,
)
print(f"\n[ok] baseline saved -> {paths.BASELINE_PATH}")
print(baseline_summary)

## 12 &middot; Run 2 &mdash; **Perceptual** (anchor-UUID mm_hash)

Same videos, same questions, same model &mdash; only difference: every
sampled frame goes through `PerceptualMMCache.prepare_image` before
vLLM sees it. Visually-similar frames now share a UUID, so the vision
encoder skips redundant work and LMCache reuses KV blocks.

In [ ]:
from eval.run_perceptual import run_perceptual_benchmark

perceptual_summary = run_perceptual_benchmark(
    videos_dir=paths.VIDEOS_DIR,
    qa_file=paths.QA_FILE,
    output_path=paths.PERCEPTUAL_PATH,
    model=MODEL,
    tau=TAU,
    k=K,
    fps=FPS,
)
print(f"\n[ok] perceptual saved -> {paths.PERCEPTUAL_PATH}")
print(perceptual_summary)

## 13 &middot; &#127919; Headline metrics &mdash; the acceleration story

The three numbers below are the hackathon pitch:

- **TTFT total** &mdash; wall-clock from `vllm.LLM.generate()`. The acceleration.
- **vision KV pressure** &mdash; (unique mm_hashes &times; per-frame tok &times; KV-bytes-per-token).
  The KV memory reduction.
- **accuracy** &mdash; gold-letter match. The parity check.

In [ ]:
import json, statistics
from pathlib import Path

def _load(p):
    return [json.loads(l) for l in Path(p).read_text(encoding='utf-8').splitlines() if l.strip()]

def _fmt_b(n):
    n = float(n)
    for u in ('B', 'KiB', 'MiB', 'GiB', 'TiB'):
        if n < 1024:
            return f'{n:.2f} {u}'
        n /= 1024
    return f'{n:.2f} PiB'

b = _load(paths.BASELINE_PATH)
p = _load(paths.PERCEPTUAL_PATH)

def s(rows, k): return sum(r.get(k, 0) for r in rows)

# Vision KV pressure: unique-frame KV bytes
def kv_pressure(rows):
    tot = 0
    for r in rows:
        mm = r.get('mm_hashes', [])
        nf = r.get('num_frames', len(mm)) or 1
        u = len(set(mm)) if mm else nf
        bpt = r.get('kv_bytes_per_token', 0)
        if bpt and r.get('prompt_tokens'):
            tot += int((r['prompt_tokens'] / nf) * u * bpt)
    return tot

b_uniq = sum(len(set(r.get('mm_hashes', []))) for r in b)
p_uniq = sum(len(set(r.get('mm_hashes', []))) for r in p)
b_press = kv_pressure(b); p_press = kv_pressure(p)
ttft_b = sum(r['ttft_s'] for r in b); ttft_p = sum(r['ttft_s'] for r in p)
acc_b = sum(1 for r in b if r.get('correct')) / max(1, len(b)) * 100
acc_p = sum(1 for r in p if r.get('correct')) / max(1, len(p)) * 100

speedup = (ttft_b - ttft_p) / ttft_b * 100 if ttft_b else 0
press_pct = (b_press - p_press) / b_press * 100 if b_press else 0

bar = '=' * 76
print(bar)
print(f"  {'METRIC':<34}{'BASELINE':>16}{'PERCEPTUAL':>16}{'WIN':>10}")
print(bar)
print(f"  {'rows':<34}{len(b):>16}{len(p):>16}{'-':>10}")
print(f"  {'total frames':<34}{s(b,'num_frames'):>16}{s(p,'num_frames'):>16}{'-':>10}")
print(f"  {'unique frames (post-alias)':<34}{b_uniq:>16}{p_uniq:>16}{b_uniq-p_uniq:>+10}")
print(f"  {'vision KV pressure':<34}{_fmt_b(b_press):>16}{_fmt_b(p_press):>16}{press_pct:>+9.1f}%")
print(f"     -> saved: {_fmt_b(b_press - p_press)}")
print(f"  {'TTFT total':<34}{ttft_b:>14.3f}s {ttft_p:>14.3f}s {speedup:>+9.1f}%")
print(f"  {'TTFT mean':<34}{statistics.mean(r['ttft_s'] for r in b):>14.3f}s "
      f"{statistics.mean(r['ttft_s'] for r in p):>14.3f}s")
print(f"  {'accuracy':<34}{acc_b:>15.1f}%{acc_p:>15.1f}%{acc_p - acc_b:>+9.1f}pp")
print(bar)

# Big headline card (plain ASCII so it renders in any terminal/output)
print()
print(f"  [*] TTFT acceleration:    {speedup:+.1f}%   ({ttft_b - ttft_p:+.2f}s faster total)")
print(f"  [*] KV pressure reduced:  {press_pct:+.1f}%  ({_fmt_b(b_press - p_press)} freed)")
print(f"  [*] Accuracy delta:       {acc_p - acc_b:+.1f}pp  "
      f"({'preserved' if abs(acc_p - acc_b) < 1 else 'CHECK -- regressed'})")

## 14 &middot; Per-video comparison &mdash; where the wins land

In [ ]:
videos = sorted({r['video'] for r in b} | {r['video'] for r in p})

def avg_per_video(rows, fn):
    out: dict[str, list[float]] = {}
    for r in rows:
        out.setdefault(r['video'], []).append(fn(r))
    return {v: (sum(xs)/len(xs) if xs else 0.0) for v, xs in out.items()}

ttft_bv = avg_per_video(b, lambda r: r['ttft_s'])
ttft_pv = avg_per_video(p, lambda r: r['ttft_s'])
uniq_bv = avg_per_video(b, lambda r: len(set(r.get('mm_hashes', []))))
uniq_pv = avg_per_video(p, lambda r: len(set(r.get('mm_hashes', []))))

x = np.arange(len(videos))
w = 0.4

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].bar(x - w/2, [ttft_bv[v] for v in videos], w, label='baseline', color='#888')
axes[0].bar(x + w/2, [ttft_pv[v] for v in videos], w, label='perceptual', color='#4a90e2')
axes[0].set_xticks(x); axes[0].set_xticklabels(videos, rotation=30, ha='right')
axes[0].set_ylabel('mean TTFT (s)')
axes[0].set_title('Per-video TTFT  (lower is faster)')
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(x - w/2, [uniq_bv[v] for v in videos], w, label='baseline', color='#888')
axes[1].bar(x + w/2, [uniq_pv[v] for v in videos], w, label='perceptual', color='#4a90e2')
axes[1].set_xticks(x); axes[1].set_xticklabels(videos, rotation=30, ha='right')
axes[1].set_ylabel('mean unique mm_hashes / call')
axes[1].set_title('Per-video unique vision-token chunks  (lower = more compression)')
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

## 15 &middot; TTFT distribution &mdash; the shift, not just the average

A histogram + CDF gives the full picture. The CDF curve for perceptual
sitting **left** of baseline is the acceleration claim, end-to-end.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

ttfts_b = [r['ttft_s'] for r in b]
ttfts_p = [r['ttft_s'] for r in p]

# Histogram
bins = 20
axes[0].hist(ttfts_b, bins=bins, alpha=0.55, label=f'baseline (n={len(ttfts_b)})', color='#888')
axes[0].hist(ttfts_p, bins=bins, alpha=0.55, label=f'perceptual (n={len(ttfts_p)})', color='#4a90e2')
axes[0].set_xlabel('TTFT (s)'); axes[0].set_ylabel('count')
axes[0].set_title('TTFT histogram')
axes[0].legend(); axes[0].grid(alpha=0.3)

# CDF
def cdf(xs):
    xs = sorted(xs)
    return xs, np.linspace(0, 1, len(xs))

xb, yb = cdf(ttfts_b)
xp, yp = cdf(ttfts_p)
axes[1].plot(xb, yb, label='baseline',   color='#888',    lw=2.5)
axes[1].plot(xp, yp, label='perceptual', color='#4a90e2', lw=2.5)
axes[1].set_xlabel('TTFT (s)'); axes[1].set_ylabel('CDF')
axes[1].set_title('TTFT CDF  (left curve = faster)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 16 &middot; Threshold &tau; sweep  *(optional, but recommended for the demo)*

Higher &tau; = stricter cosine match = fewer aliases but safer
(lower false-positive rate). Lower &tau; = more aggressive aliasing,
risk of false positives. The sweep finds the highest hit-rate &tau;
that doesn't regress accuracy below `baseline_accuracy - 1pp`.

In [ ]:
from eval.threshold_sweep import sweep_thresholds

sweep_thresholds(
    videos_dir=paths.VIDEOS_DIR,
    qa_file=paths.QA_FILE,
    taus=[0.95, 0.96, 0.97, 0.98, 0.99],
    output_dir=paths.SWEEP_DIR,
    model=MODEL,
    baseline_path=paths.BASELINE_PATH,
)
print(f"\n[ok] sweep -> {paths.SWEEP_DIR}")

## 17 &middot; Generate `REPORT.md` (full metric table + plots)

In [ ]:
from eval.benchmark_videoqa import analyze_results

out = analyze_results(
    baseline_path=paths.BASELINE_PATH,
    perceptual_path=paths.PERCEPTUAL_PATH,
    sweep_dir=paths.SWEEP_DIR,
    output_dir=paths.REPORT_DIR,
)
print(f"\n[ok] report saved -> {out['report_path']}")
print(f"     plots:           {out['plots_written']}")

In [ ]:
from IPython.display import Markdown, display, Image
import os

with open(f'{paths.REPORT_DIR}/REPORT.md', encoding='utf-8') as f:
    display(Markdown(f.read()))

for png in sorted(os.listdir(paths.REPORT_DIR)):
    if png.endswith('.png'):
        print(f"\n=== {png} ===")
        display(Image(filename=f'{paths.REPORT_DIR}/{png}'))

---

## What this run demonstrated

| Track signal | Mechanism | Where to look |
|---|---|---|
| **&#9889; TTFT acceleration** | Vision encoder skips aliased frames because vLLM's `MultiModalHasher` sees the same UUID for visually-similar inputs &rarr; LMCache exact-match path serves them | section 13 (headline), section 15 (CDF shift) |
| **&#128190; KV memory reduction** | Aliased frames collapse to one `mm_hash` &rarr; LMCache stores one KV block per anchor instead of one per frame | section 13 (vision KV pressure row), section 14 (per-video unique chunks) |
| **&#127919; Quality preservation** | DINOv2-small cosine &ge; &tau; guarantees aliased frames are visually indistinguishable to the VLM | section 13 (accuracy parity) |

### Knobs you can tune
- **`TAU`** (cosine threshold, default `0.98`) &mdash; increase if
  accuracy regresses; decrease for more aggressive aliasing.
- **`K`** (pHash Hamming cap, default `5`) &mdash; widen for more BK-tree
  candidates per frame (slightly slower Tier-1, higher recall).
- **`FPS`** &mdash; higher FPS &rarr; more redundant frames &rarr; bigger pmcache win.

### What `pmcache` is **not**
- It does not subclass `LMCacheConnectorV1` or monkeypatch vLLM
  internals &mdash; it uses vLLM's public EXIF `ImageID` extension point.
- It does not change the model, the prompt, or the sampling. Same
  outputs, same accuracy, fewer GPU cycles + less KV memory.

---

*Generated by `scripts/build_colab_notebook.py`. Re-run that script
after editing source cells &mdash; the notebook is the artifact, the
builder is the source of truth.*